## 1. Configuración Inicial de Parámetros

Ejecuta la siguiente celda para configurar los parámetros de auditoría. 
Todos los valores están documentados con comentarios inline para facilitar inspección antes del envío.

In [1]:
from pathlib import Path
import sys

# Localizar el soporte tanto desde la carpeta del cuaderno como desde el repo.
SUPPORT_DIR = next((root for root in (Path.cwd(), Path.cwd() / "auditoria_IA",
    Path.cwd() / "plan_v2" / "auditoria_IA") if (root / "auditoria_config.py").is_file()), None)
if SUPPORT_DIR is None:
    raise FileNotFoundError("Abre el cuaderno desde su carpeta o desde la raiz del repositorio.")
sys.path.insert(0, str(SUPPORT_DIR.resolve()))
from auditoria_config import AuditConfig

# Cambia solo esta ruta para auditar PDF, TXT, MD o DOCX; no enviar codigo LaTeX crudo.
INPUT_DOCUMENT = SUPPORT_DIR.parent / "plan_tesis_cables_pinn.pdf"
OUTPUT_DIR = SUPPORT_DIR / "auditoria_resultados"
API_URL = "http://127.0.0.1:8000"  # Solo localhost; el documento no se envia a una nube.
CONTAINER_NAME = "sloptotal"
START_CONTAINER = True  # Arranca el existente; no recrea ni elimina contenedores/volumenes.
STARTUP_TIMEOUT = 60

ANALYSIS_MODE = "both"  # whole, chunks o both; usa el mismo endpoint para ambos.
CHUNK_MODE = "structure"  # Respeta encabezados/parrafos; words usa ventanas por palabras.
MAX_WORDS = 400  # Compromiso de contexto y coste; no es una garantia de calibracion.
MAX_CHARS = 5000  # Limite adicional; nunca se fragmenta una palabra.
MAX_CHUNKS = None  # None revisa todos; un limite produce una auditoria parcial.
REVIEW_THRESHOLD = 65.0  # Prioridad de lectura, no porcentaje de autoria IA.
REQUEST_TIMEOUT = 180  # Espera de cada POST; no reintentar tras timeout automaticamente.
REPORT_TIMEOUT = 300  # Espera adicional de un ticket, si el servidor lo devuelve.
STOP_AFTER_TIMEOUTS = 1  # Evitar acumular trabajos que sigan activos en el servidor.
DRY_RUN = False  # True prueba extraccion/informe sin red ni Docker.
SMOKE_TEST = True  # Inicio seguro: global + fragmento de una muestra, NO del PDF completo.
SMOKE_CHUNKS = 1  # Para auditoria completa cambia SMOKE_TEST a False.

config = AuditConfig(
    input_path=INPUT_DOCUMENT, output_dir=OUTPUT_DIR, api_url=API_URL,
    container_name=CONTAINER_NAME, start_container=START_CONTAINER,
    startup_timeout=STARTUP_TIMEOUT, mode=ANALYSIS_MODE, chunk_mode=CHUNK_MODE,
    max_words=MAX_WORDS, max_chars=MAX_CHARS, max_chunks=MAX_CHUNKS,
    request_timeout=REQUEST_TIMEOUT, report_timeout=REPORT_TIMEOUT,
    review_threshold=REVIEW_THRESHOLD, dry_run=DRY_RUN,
    stop_after_timeouts=STOP_AFTER_TIMEOUTS,
)
print(f"Entrada: {config.input_path}")
print(f"Modo: {config.mode}; fragmentacion: {config.chunk_mode}; prueba de humo: {SMOKE_TEST}")
print(f"Salida: {config.output_dir}")

Entrada: c:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\plan_tesis_cables_pinn.pdf
Modo: both; fragmentacion: structure; prueba de humo: True
Salida: c:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\auditoria_IA\auditoria_resultados


## 2. Importaciones y Verificación de Entorno

In [2]:
import json
from dataclasses import replace
from auditoria_config import prepare_document, diagnose_service, run_audit
from documentos import TextBlock
from auditar_plan import get_json

print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"Soporte: {SUPPORT_DIR.resolve()}")
print("Formatos: PDF con texto, TXT, Markdown y DOCX; un PDF escaneado requiere OCR previo.")

Python: 3.12.9 (c:\Python312\python.exe)
Soporte: C:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\auditoria_IA
Formatos: PDF con texto, TXT, Markdown y DOCX; un PDF escaneado requiere OCR previo.


## 3. Diagnóstico de Servicio

Verifica el estado de Docker y la API sloptotal. En `DRY_RUN=True`, este paso devuelve None sin efectos colaterales.

In [4]:
diagnostics = None
if config.dry_run:
    print("Modo seco: sin solicitudes de red ni arranque de Docker.")
else:
    diagnostics = diagnose_service(config)
    docker = diagnostics.get("docker", {})
    health = diagnostics.get("health")
    print(f"Contenedor existente: {docker.get('exists', False)}")
    print(f"Estado registrado: {docker.get('state', {}).get('Status', 'desconocido')}")
    if docker.get("start_output"):
        print(f"Arranque: {docker['start_output']}")
    if health:
        engines = health.get("engines", {})
        print(f"API: {health.get('status', 'desconocido')}; motores: {engines.get('loaded', '?')}/{engines.get('total', '?')}")
        print(f"Motores fallidos: {', '.join(engines.get('failed', [])) or 'ninguno reportado'}")
        diagnostics["queue"] = get_json(config.api_url, "/api/queue/status")
        print("Colas:", json.dumps(diagnostics["queue"], ensure_ascii=False))
    else:
        raise RuntimeError(f"API no disponible: {docker.get('health_error', docker.get('error', 'sin detalles'))}")

Diagnosticando servicio...

✓ Estado Docker:
  - Disponible: True
  - Existe contenedor 'sloptotal': True
  - En ejecución: True

✓ Salud API:
  - Status: healthy
  - Motores cargados: 12/23
  - Motores fallidos: perplexity, cross_perplexity, classifier_openai, classifier_chatgpt, classifier_remodetect, classifier_fakespot, classifier_e5, classifier_tmr, classifier_bert_raid, classifier_desklib, classifier_superannotate

✓ Estado Docker:
  - Disponible: True
  - Existe contenedor 'sloptotal': True
  - En ejecución: True

✓ Salud API:
  - Status: healthy
  - Motores cargados: 12/23
  - Motores fallidos: perplexity, cross_perplexity, classifier_openai, classifier_chatgpt, classifier_remodetect, classifier_fakespot, classifier_e5, classifier_tmr, classifier_bert_raid, classifier_desklib, classifier_superannotate


## 4. Extracción y Vista Previa del Documento

In [5]:
full_document, chunks = prepare_document(config)
document = full_document
print(f"Formato: {document.metadata.get('format', config.input_path.suffix)}")
print(f"Texto: {len(document.text.split()):,} palabras; {len(document.text):,} caracteres")
print(f"Fragmentos seleccionados: {len(chunks)}/{document.metadata.get('audit_total_chunks', len(chunks))}")
for warning in document.warnings:
    print(f"Advertencia: {warning}")

if SMOKE_TEST:
    if SMOKE_CHUNKS < 1 or not chunks:
        raise ValueError("La prueba necesita al menos un fragmento.")
    chunks = chunks[:SMOKE_CHUNKS]
    sample = "\n\n".join(chunk.text for chunk in chunks)
    document = replace(full_document, text=sample,
        blocks=[TextBlock(sample, "muestra de prueba de humo")],
        metadata={**full_document.metadata, "smoke_test": True,
            "source_document_words": len(full_document.text.split())},
        warnings=[*full_document.warnings, "PRUEBA DE HUMO: global corresponde SOLO a la muestra, no al documento completo."])
    print(f"PRUEBA DE HUMO: {len(sample.split())} palabras; {len(chunks)} fragmento(s). Global = muestra.")

if chunks:
    first = chunks[0]
    print(f"Primer fragmento: {first.chunk_id}; ubicaciones: {', '.join(first.locations)}")
    print(f"Secciones: {', '.join(first.sections) or 'no detectadas'}")
    print(first.text[:500])
assert all(len(chunk.text.split()) <= config.max_words and len(chunk.text) <= config.max_chars for chunk in chunks)
print("Limites de fragmentacion comprobados.")

Extrayendo documento...

✓ Extracción completada:
  - Total de páginas en PDF: 73
  - Cuerpo detectado: páginas 11–60
  - Palabras en cuerpo: 17,787
  - Caracteres en cuerpo: 119,150
  - Fragmentos generados: 50

Primer fragmento (primeros 500 caracteres):
  ID: p011_c001
  Página: 11
  Capítulo: CAPÍTULO 1. PLANTEAMIENTO DEL PROBLEMA
  Sección: 1.1 INTRODUCCIÓN

CAPÍTULO 1. PLANTEAMIENTO DEL PROBLEMA
1.1 INTRODUCCIÓN
En el Perú, entre 2010 y 2025, la máxima demanda del Sistema Eléctrico Interconectado
Nacional (SEIN) aumentó de 4579 MW a 7942 MW. En el mismo período, la participación
conjunta de la generación solar y eólica pasó de un valor cercano a cero a 9,5 %. Estos
cambios aumentan la exigencia de transportar energía y aprovechar la capacidad disponible
de la red (Ministerio de Energía y Minas, 2025; Ministerio de Energía y Minas, 2026b; COES,
2026…


## 5. Ejecución de Auditoría

Se enviarán requests a la API según los parámetros configurados. 
El progreso se mostrará en tiempo real.

In [ ]:
print("Iniciando auditoría...")
print(f"Modo: {config.mode} | Modo de fragmentación: {config.chunk_mode}")

try:
    report_path = run_audit(config, document, chunks, diagnostics)
    print(f"\n✓ Auditoría completada exitosamente")
    print(f"Reporte disponible en: {report_path}")
    
    # Mostrar tamaño del directorio de salida
    output_files = list(config.output_dir.glob("*"))
    print(f"\nArtefactos generados ({len(output_files)} archivo(s)):")
    for f in sorted(output_files)[:10]:
        size_bytes = f.stat().st_size if f.is_file() else 0
        size_kb = size_bytes / 1024
        print(f"  - {f.name} ({size_kb:.1f} KB)")
    if len(output_files) > 10:
        print(f"  ... y {len(output_files) - 10} archivo(s) más")
        
except Exception as e:
    print(f"\n✗ Error durante auditoría:")
    print(f"  {type(e).__name__}: {str(e)}")
    report_path = None


Iniciando auditoría...
Modo: both | Modo de fragmentación: structure


## 6. Visualización de Reporte

Se muestra el reporte Markdown completo con resultados y análisis.

In [ ]:
if report_path and report_path.is_file():
    report_content = report_path.read_text(encoding="utf-8")
    # Mostrar como Markdown en el notebook
    from IPython.display import Markdown, display
    display(Markdown(report_content))
else:
    print("⚠ No se pudo cargar el reporte. Verifica que la auditoría haya completado exitosamente.")


# Auditoría Sloptotal: Detección de IA en Documentos

**Versión**: 1.0  
**Última actualización**: 2026-11-13  
**Descripción**: Flujo reproducible de auditoría de documentos con sloptotal local.

Este cuaderno automatiza la evaluación de documentos contra un servicio de detección de IA usando la API local de sloptotal. Todas las configuraciones están centralizadas al inicio para facilitar la inspección antes del envío.

## Características

- ✓ Soporte para PDF con extracción de cuerpo de documento
- ✓ Extracción preservando texto completo (portada, referencias, bibliografía se excluyen)
- ✓ Segmentación por caracteres para control granular
- ✓ Diagnóstico Docker e inicio de contenedor existente
- ✓ Modo seco (dry_run=True) para inspeccionar antes de enviar
- ✓ Puntuación global y por fragmentos secuencialmente en `/api/analyze`
- ✓ Reportes aislados por ejecución con entrada exacta y artefactos (json, manifest)

## Límites de interpretación

- Una puntuación de detector no establece quién escribió un pasaje
- La puntuación global y las puntuaciones por fragmento responden a entradas distintas
- Los comentarios heurísticos solo señalan carga sintáctica y repetición de conectores
- Para interpretar una alerta, contrastarla con historial de versiones y decisiones de redacción